## import data and packages

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from numpy import genfromtxt
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy.stats import linregress
from matplotlib.colorbar import ColorbarBase
import matplotlib.colors as mcolors
import matplotlib.cm as cm

In [ ]:
scaled_input_ohc_train = genfromtxt('ohc700_train_robustscaler.csv', delimiter=',')
scaled_input_ohc_test = genfromtxt('ohc700_test_robustscaler.csv', delimiter=',')

scaled_input_sst_train = genfromtxt('sst_train_robustscaler.csv', delimiter=',')
scaled_input_sst_test = genfromtxt('sst_test_robustscaler.csv', delimiter=',')

scaled_input_olr_train = genfromtxt('olr_train_robustscaler.csv', delimiter=',')
scaled_input_olr_test = genfromtxt('olr_test_robustscaler.csv', delimiter=',')

In [ ]:
# load netcdf files with time component 
sequential_input_ohc_train = xr.load_dataarray('ohc700_train_robustscaler.nc')
sequential_input_ohc_test = xr.load_dataarray('ohc700_test_robustscaler.nc')

sequential_input_olr_train = xr.load_dataarray('olr_train_robustscaler.nc')
sequential_input_olr_test = xr.load_dataarray('olr_test_robustscaler.nc')

sequential_input_sst_train = xr.load_dataarray('sst_train_robustscaler.nc')
sequential_input_sst_test = xr.load_dataarray('sst_test_robustscaler.nc')

In [ ]:
# sanity checks
print(scaled_input_sst_train.shape,
     scaled_input_sst_test.shape,
     scaled_input_ohc_train.shape,
     scaled_input_ohc_test.shape,
     scaled_input_olr_train.shape,
     scaled_input_olr_test.shape)

In [ ]:
# Concatenate the train and test datasets along the 'time' dimension
ohc_combined = xr.concat([sequential_input_ohc_train, sequential_input_ohc_test], dim='time')
olr_combined = xr.concat([sequential_input_olr_train, sequential_input_olr_test], dim='time')
sst_combined = xr.concat([sequential_input_sst_train, sequential_input_sst_test], dim='time')

In [ ]:
# use sort by for sequential data along time dimension
ohc_sequential = ohc_combined.sortby('time')
olr_sequential = olr_combined.sortby('time')
sst_sequential = sst_combined.sortby('time')

print(sst_sequential.shape,
      ohc_sequential.shape,
      olr_sequential.shape
     )

In [ ]:
latent_representations = xr.open_dataset('latent_representations_zmean.nc')

In [ ]:
oni_e3sm = xr.open_dataset('oni_e3sm_0402.nc')

In [ ]:
scaler_sst = joblib.load('scaler_sst.pkl')

In [ ]:
ocn = xr.open_dataset('ocean.nc')
atm = xr.open_dataset('atmosphere.nc')

ocn_lat = ocn.lat.values
ocn_lon = ocn.lon.values

atm_lat = atm.lat.values
atm_lon = atm.lon.values

In [ ]:
# equatorial ocean mask
ocn_lat_mask = (ocn_lat >= -3) & (ocn_lat <= 3)

In [ ]:
# grid reconstruction for ocean
def reconstruct_grid(pred_1d, y_indx, x_indx):
    """
    Reconstruct the original 2d grid using NN predictions.
    Args:
        pred_1d (xarray data array): predictions (samples, 1d values)
        y_indx (numpy array): latitude indices for grid (20)
        x_indx (numpy array): longitude indices for grid (150)
    Returns:
        pred_grid (numpy array): predictions on original 2d grid
    """
    # number of predicted samples (!assuming axis=0!)
    n_samples = pred_1d.shape[0]
    
    # grid to be populated with predicted data
    pred_grid = np.full((n_samples, 20, 150), np.nan)
    
    # fill the reconstructed grid
    pred_grid[:, y_indx, x_indx] = pred_1d#.values
    
    # return predictions on original 2d grid
    return pred_grid

In [ ]:
el_nino_filtered = pd.read_pickle('elnino_events_filtered.pkl')
la_nina_filtered = pd.read_pickle('lanina_events_filtered.pkl')

In [ ]:
# empty arrays
peak_lon_nino_events = []
peak_lon_nina_events = []
latent_nino_events = []
latent_nina_events = []

## El Niño events

In [ ]:
# El Niño events
for _, row in el_nino_filtered.iterrows():
    month_indices = np.arange(row['start_idx'], row['end_idx'] + 1)
    
    # SST peak longitude — with inverse transform
    sst_event = sst_sequential[month_indices]  # scaled, shape (n_months, 2968)
    
    # Apply inverse transform to get physical SST anomaly values
    sst_event_physical = scaler_sst.inverse_transform(sst_event.values)  # (n_months, 2968)
    
    sst_event_grid = reconstruct_grid(sst_event_physical, sst_yindx, sst_xindx)
    sst_event_mean = np.nanmean(sst_event_grid, axis=0)
    sst_equator = sst_event_mean[ocn_lat_mask, :]
    zonal_mean = np.nanmean(sst_equator, axis=0)
    
    if np.all(np.isnan(zonal_mean)):
        print(f"Warning: all NaN for El Niño event at index {row['start_idx']}-{row['end_idx']}")
        peak_lon_nino_events.append(np.nan)
        latent_nino_events.append(np.full(20, np.nan))
        continue
        
    peak_idx = np.nanargmax(zonal_mean)
    peak_lon_nino_events.append(ocn_lon[peak_idx])
    
    # Latent event mean — unchanged, stays in latent space
    latent_event_mean = np.nanmean(latent_representations['z_mean'].values[month_indices], axis=0)
    latent_nino_events.append(latent_event_mean)

peak_lon_nino_events = np.array(peak_lon_nino_events)
latent_nino_events = np.array(latent_nino_events)  # (92, 20)

print(f"El Niño: {peak_lon_nino_events.shape}, {latent_nino_events.shape}")

def lon_formatter(x, pos):
    if x <= 180:
        return f"{int(x)}°E"
    else:
        return f"{int(360 - x)}°W"

# Blob threshold: 140°W = 220° in 0-360
LON_THRESHOLD = 220

cp_mask = peak_lon_nino_events <= LON_THRESHOLD  # Central Pacific blob
ep_mask = peak_lon_nino_events > LON_THRESHOLD   # Eastern Pacific blob
print(f"CP events: {cp_mask.sum()}, EP events: {ep_mask.sum()}")

# --- Compute peak SST value at peak longitude for each El Niño event (physical units) ---
peak_sst_nino = []
for i, (_, row) in enumerate(el_nino_filtered.iterrows()):
    month_indices = np.arange(row['start_idx'], row['end_idx'] + 1)
    sst_event = sst_sequential[month_indices]
    sst_event_physical = scaler_sst.inverse_transform(sst_event.values)
    sst_event_grid = reconstruct_grid(sst_event_physical, sst_yindx, sst_xindx)
    sst_event_mean = np.nanmean(sst_event_grid, axis=0)
    sst_equator = sst_event_mean[ocn_lat_mask, :]
    zonal_mean = np.nanmean(sst_equator, axis=0)
    if np.all(np.isnan(zonal_mean)):
        peak_sst_nino.append(np.nan)
    else:
        peak_idx = np.nanargmax(zonal_mean)
        peak_sst_nino.append(zonal_mean[peak_idx])

peak_sst_nino = np.array(peak_sst_nino)

In [ ]:
# plotting
vmin =  0 #np.nanmin(peak_sst_nino) # if you want to be safe
vmax = np.nanpercentile(peak_sst_nino, 95)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = cm.Reds 

n_latent = latent_nino_events.shape[1]
nrows, ncols = 5, 4
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(13, 12),
    sharex=True
)
fig.supxlabel("Peak SST Longitude", fontsize=18, y=-0.005)
fig.supylabel(r"Latent Posterior Mean ($\mu$)", fontsize=18, x=-0.005)
plt.subplots_adjust(
    left=0.06, right=0.96, bottom=0.08, top=0.92,
    wspace=0.15, hspace=0.20
)
axes = axes.flatten()

for d in range(n_latent):
    ax = axes[d]
    ax.grid(True)
    ax.set_ylim(-3, 3)
    ax.set_yticks([-2, -1, 0, 1, 2])
    x = peak_lon_nino_events
    y = latent_nino_events[:, d]
    valid = ~(np.isnan(x) | np.isnan(y) | np.isnan(peak_sst_nino))

    # Scatter colored by peak SST magnitude
    sc = ax.scatter(
        x[valid], y[valid],
        c=peak_sst_nino[valid],
        cmap=cmap, norm=norm,
        s=60, alpha=0.8,
        edgecolors='gray', linewidths=0.3
    )

    ax.axvline(LON_THRESHOLD, color='gray', linewidth=0.8, linestyle='--')

    # Regression lines — all, CP, EP
    if valid.sum() > 2:
        slope, intercept, r_all, p_all, _ = linregress(x[valid], y[valid])
        x_line = np.linspace(x[valid].min(), x[valid].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='black', linewidth=1.5)
        sig_all = '*' if p_all < 0.05 else ''
        ax.text(0.45, 0.95, f'r$_{{all}}$={r_all:.2f}{sig_all}',
                transform=ax.transAxes, fontsize=12,
                verticalalignment='top', color='black')

    cp_valid = valid & cp_mask
    if cp_valid.sum() > 2:
        slope, intercept, r_cp, p_cp, _ = linregress(x[cp_valid], y[cp_valid])
        x_line = np.linspace(x[cp_valid].min(), x[cp_valid].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='blue', linewidth=1.5)
        sig_cp = '*' if p_cp < 0.05 else ''
        ax.text(0.05, 0.95, f'r$_{{CP}}$={r_cp:.2f}{sig_cp}',
                transform=ax.transAxes, fontsize=12,
                verticalalignment='top', color='blue')

    ep_valid = valid & ep_mask
    if ep_valid.sum() > 2:
        slope, intercept, r_ep, p_ep, _ = linregress(x[ep_valid], y[ep_valid])
        x_line = np.linspace(x[ep_valid].min(), x[ep_valid].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='green', linewidth=1.5)
        sig_ep = '*' if p_ep < 0.05 else ''
        ax.text(0.05, 0.8, f'r$_{{EP}}$={r_ep:.2f}{sig_ep}',
                transform=ax.transAxes, fontsize=12,
                verticalalignment='top', color='green')

    ax.set_title(f"LD{d+1}", fontsize=14)
    row = d // ncols
    col = d % ncols
    if col == 0:
        ax.tick_params(axis='y', labelsize=12)
    else:
        ax.set_yticklabels([])

    # Set xlim and xticks for ALL panels
    ax.set_xlim(120, 290)
    ax.set_xticks([130, 160, 190, 220, 250, 280])

    if row == nrows - 1:
        ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
        ax.tick_params(axis="x", labelrotation=45, labelsize=14)
    else:
        ax.set_xticklabels([])

for ax in axes[n_latent:]:
    ax.remove()

# Shared colorbar
cbar_ax = fig.add_axes([0.97, 0.08, 0.02, 0.84])
cb = ColorbarBase(cbar_ax, cmap=cmap, norm=norm, extend='max', orientation='vertical')
cb.set_label('Peak SST Anomaly (°C)', fontsize=14)

#plt.savefig('peaklonnino_withmarkers_0315_v2.pdf', bbox_inches='tight')
plt.show()

## La Niña events

In [ ]:
# La Niña events
for _, row in la_nina_filtered.iterrows():
    month_indices = np.arange(row['start_idx'], row['end_idx'] + 1)
    
    # SST peak longitude — with inverse transform
    sst_event = sst_sequential[month_indices]  # scaled, shape (n_months, 2968)
    
    # Apply inverse transform to get physical SST anomaly values
    sst_event_physical = scaler_sst.inverse_transform(sst_event.values)  # (n_months, 2968)
    
    sst_event_grid = reconstruct_grid(sst_event_physical, sst_yindx, sst_xindx)
    sst_event_mean = np.nanmean(sst_event_grid, axis=0)
    sst_equator = sst_event_mean[ocn_lat_mask, :]
    zonal_mean = np.nanmean(sst_equator, axis=0)
    
    if np.all(np.isnan(zonal_mean)):
        print(f"Warning: all NaN for La Niña event at index {row['start_idx']}-{row['end_idx']}")
        peak_lon_nina_events.append(np.nan)
        latent_nina_events.append(np.full(20, np.nan))
        continue
        
    peak_idx = np.nanargmin(zonal_mean)
    peak_lon_nina_events.append(ocn_lon[peak_idx])
    
    # Latent event mean — unchanged, stays in latent space
    latent_event_mean = np.nanmean(latent_representations['z_mean'].values[month_indices], axis=0)
    latent_nina_events.append(latent_event_mean)

peak_lon_nina_events = np.array(peak_lon_nina_events)
latent_nina_events = np.array(latent_nina_events)  # (92, 20)

print(f"La Niña: {peak_lon_nina_events.shape}, {latent_nina_events.shape}")

def lon_formatter(x, pos):
    if x <= 180:
        return f"{int(x)}°E"
    else:
        return f"{int(360 - x)}°W"

LON_THRESHOLD = 220
cp_mask_nina = peak_lon_nina_events <= LON_THRESHOLD
ep_mask_nina = peak_lon_nina_events > LON_THRESHOLD

# --- Compute minimum SST value at peak (cold) longitude for each La Niña event ---
peak_sst_nina = []
for i, (_, row) in enumerate(la_nina_filtered.iterrows()):
    month_indices = np.arange(row['start_idx'], row['end_idx'] + 1)
    sst_event = sst_sequential[month_indices]
    sst_event_physical = scaler_sst.inverse_transform(sst_event.values)
    sst_event_grid = reconstruct_grid(sst_event_physical, sst_yindx, sst_xindx)
    sst_event_mean = np.nanmean(sst_event_grid, axis=0)
    sst_equator = sst_event_mean[ocn_lat_mask, :]
    zonal_mean = np.nanmean(sst_equator, axis=0)
    if np.all(np.isnan(zonal_mean)):
        peak_sst_nina.append(np.nan)
    else:
        # argmin here, not argmax like Niño
        peak_idx = np.nanargmin(zonal_mean)  # cold anomaly
        peak_sst_nina.append(zonal_mean[peak_idx])

peak_sst_nina = np.array(peak_sst_nina)

In [ ]:
# plotting
# colormap — cold anomalies, expect negative values
vmin = np.nanpercentile(peak_sst_nina, 5)  # most negative end
vmax = 0
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = cm.Blues_r  # dark blue = most negative/cold

n_latent = latent_nina_events.shape[1]
nrows, ncols = 5, 4
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(13, 12),
    sharex=True
)
fig.supxlabel("Minimum SST Longitude", fontsize=18, y=-0.005)
fig.supylabel(r"Latent Posterior Mean ($\mu$)", fontsize=18, x=-0.005)
plt.subplots_adjust(
    left=0.06, right=0.96, bottom=0.08, top=0.92,
    wspace=0.15, hspace=0.20
)
axes = axes.flatten()

for d in range(n_latent):
    ax = axes[d]
    ax.grid(True)
    ax.set_ylim(-3, 3)
    ax.set_yticks([-2, -1, 0, 1, 2])
    x = peak_lon_nina_events
    y = latent_nina_events[:, d]
    valid = ~(np.isnan(x) | np.isnan(y) | np.isnan(peak_sst_nina))

    # Scatter colored by minimum SST magnitude
    sc = ax.scatter(
        x[valid], y[valid],
        c=peak_sst_nina[valid],
        cmap=cmap, norm=norm,
        s=60, alpha=0.8,
        edgecolors='gray', linewidths=0.3
    )

    ax.axvline(LON_THRESHOLD, color='gray', linewidth=0.8, linestyle='--')

    # Regression lines — all, CP, EP
    if valid.sum() > 2:
        slope, intercept, r_all, p_all, _ = linregress(x[valid], y[valid])
        x_line = np.linspace(x[valid].min(), x[valid].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='black', linewidth=1.5)
        sig_all = '*' if p_all < 0.05 else ''
        ax.text(0.45, 0.95, f'r$_{{all}}$={r_all:.2f}{sig_all}',
                transform=ax.transAxes, fontsize=12,
                verticalalignment='top', color='black')

    cp_valid = valid & cp_mask_nina
    if cp_valid.sum() > 2:
        slope, intercept, r_cp, p_cp, _ = linregress(x[cp_valid], y[cp_valid])
        x_line = np.linspace(x[cp_valid].min(), x[cp_valid].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='blue', linewidth=1.5)
        sig_cp = '*' if p_cp < 0.05 else ''
        ax.text(0.05, 0.95, f'r$_{{CP}}$={r_cp:.2f}{sig_cp}',
                transform=ax.transAxes, fontsize=12,
                verticalalignment='top', color='blue')

    ep_valid = valid & ep_mask_nina
    if ep_valid.sum() > 2:
        slope, intercept, r_ep, p_ep, _ = linregress(x[ep_valid], y[ep_valid])
        x_line = np.linspace(x[ep_valid].min(), x[ep_valid].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='green', linewidth=1.5)
        sig_ep = '*' if p_ep < 0.05 else ''
        ax.text(0.05, 0.8, f'r$_{{EP}}$={r_ep:.2f}{sig_ep}',
                transform=ax.transAxes, fontsize=12,
                verticalalignment='top', color='green')

    ax.set_title(f"LD{d+1}", fontsize=14)
    row = d // ncols
    col = d % ncols
    if col == 0:
        ax.tick_params(axis='y', labelsize=12)
    else:
        ax.set_yticklabels([])

    # Set xlim and xticks for ALL panels
    ax.set_xlim(120, 290)
    ax.set_xticks([130, 160, 190, 220, 250, 280])

    if row == nrows - 1:
        ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
        ax.tick_params(axis="x", labelrotation=45, labelsize=14)
    else:
        ax.set_xticklabels([])

for ax in axes[n_latent:]:
    ax.remove()

# Shared colorbar
cbar_ax = fig.add_axes([0.97, 0.08, 0.02, 0.84])
cb = ColorbarBase(cbar_ax, cmap=cmap, norm=norm, extend='min', orientation='vertical')
cb.set_label('Minimum SST Anomaly (°C)', fontsize=14)

#plt.savefig('peaklonnina_withmarkers_0315_v2.pdf', bbox_inches='tight')
plt.show()